In [0]:
from pyspark.sql.functions import *

### reading the claen silver data in gold Schema

In [0]:
posts_df = spark.table("de_project.silver.posts_clean")
comments_df = spark.table("de_project.silver.comments_clean")

### Standardize both tables for union

In [0]:
posts_final = posts_df.select(
    col("id").alias("log_id"),
    col("user_id").cast("string").alias("user_id"),
    col("log_timestamp"),
    col("log_date"),
    col("log_hour"),
    col("log_level"),
    col("is_error"),
    col("body_clean"),
    lit("post").alias("source_type"),
    col("Current_timestamp")
)

comments_final = comments_df.select(
    col("id").alias("log_id"),
    lit(None).cast("string").alias("user_id"),
    col("log_timestamp"),
    col("log_date"),
    col("log_hour"),
    col("log_level"),
    col("is_error"),
    col("body_clean"),
    lit("comment").alias("source_type"),
    col("Current_timestamp")
)

### performing union

In [0]:
fact_log_detail = posts_final.unionByName(comments_final)

In [0]:
display(fact_log_detail)

log_id,user_id,log_timestamp,log_date,log_hour,log_level,is_error,body_clean,source_type,Current_timestamp
1,1,2026-04-06T01:49:27.863462Z,2026-04-06,1,INFO,0,quia et suscipit suscipit recusandae consequuntur expedita et cum reprehenderit molestiae ut ut quas totam nostrum rerum est autem sunt rem eveniet architecto,post,2026-04-06T14:54:32.043401Z
2,1,2026-04-05T23:00:57.069687Z,2026-04-05,23,INFO,0,est rerum tempore vitae sequi sint nihil reprehenderit dolor beatae ea dolores neque fugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis qui aperiam non debitis possimus qui neque nisi nulla,post,2026-04-06T14:54:32.043401Z
3,1,2026-04-06T06:16:08.914975Z,2026-04-06,6,INFO,0,et iusto sed quo iure voluptatem occaecati omnis eligendi aut ad voluptatem doloribus vel accusantium quis pariatur molestiae porro eius odio et labore et velit aut,post,2026-04-06T14:54:32.043401Z
4,1,2026-04-06T11:48:37.691745Z,2026-04-06,11,INFO,0,ullam et saepe reiciendis voluptatem adipisci sit amet autem assumenda provident rerum culpa quis hic commodi nesciunt rem tenetur doloremque ipsam iure quis sunt voluptatem rerum illo velit,post,2026-04-06T14:54:32.043401Z
5,1,2026-04-05T20:31:13.61473Z,2026-04-05,20,INFO,0,repudiandae veniam quaerat sunt sed alias aut fugiat sit autem sed est voluptatem omnis possimus esse voluptatibus quis est aut tenetur dolor neque,post,2026-04-06T14:54:32.043401Z
6,1,2026-04-06T14:30:39.622761Z,2026-04-06,14,INFO,0,ut aspernatur corporis harum nihil quis provident sequi mollitia nobis aliquid molestiae perspiciatis et ea nemo ab reprehenderit accusantium quas voluptate dolores velit et doloremque molestiae,post,2026-04-06T14:54:32.043401Z
7,1,2026-04-05T15:17:19.993093Z,2026-04-05,15,INFO,0,dolore placeat quibusdam ea quo vitae magni quis enim qui quis quo nemo aut saepe quidem repellat excepturi ut quia sunt ut sequi eos ea sed quas,post,2026-04-06T14:54:32.043401Z
8,1,2026-04-06T08:30:27.972111Z,2026-04-06,8,INFO,0,dignissimos aperiam dolorem qui eum facilis quibusdam animi sint suscipit qui sint possimus cum quaerat magni maiores excepturi ipsam ut commodi dolor voluptatum modi aut vitae,post,2026-04-06T14:54:32.043401Z
9,1,2026-04-05T19:01:23.300768Z,2026-04-05,19,INFO,0,consectetur animi nesciunt iure dolore enim quia ad veniam autem ut quam aut nobis et est aut quod aut provident voluptas autem voluptas,post,2026-04-06T14:54:32.043401Z
10,1,2026-04-06T04:50:10.149086Z,2026-04-06,4,ERROR,1,quo et expedita modi cum officia vel magni doloribus qui repudiandae vero nisi sit quos veniam quod sed accusamus veritatis error,post,2026-04-06T14:54:32.043401Z


### top domains with errors

In [0]:
domain_errors = (
    comments_df
    .groupBy("email_domain")
    .agg(
        count("*").alias("total_comments"),
        sum("is_error").alias("error_comments")
    )
    .withColumn("error_rate", col("error_comments") / col("total_comments"))
    .orderBy(col("error_comments").desc())
)

In [0]:
#Counts logs by domain and level

domain_behavior = (
    comments_df
    .groupBy("email_domain", "log_level")
    .count()
)

In [0]:
#Aggregates error count by domain hour

email_time = (
    comments_df
    .groupBy("email_domain", "log_hour")
    .agg(sum("is_error").alias("error_count"))
)

### filters users with high error rate

In [0]:
suspicious_users = (
    comments_df
    .groupBy("email_clean")
    .agg(
        count("*").alias("total"),
        sum("is_error").alias("errors")
    )
    .withColumn("error_rate", col("errors") / col("total"))
    .filter(col("error_rate") > 0.5)
)

### create daily summary

In [0]:
fact_log_summary = (
    fact_log_detail.groupBy("log_date")
    .agg(
        count("*").alias("total_logs"),
        sum(when(col("log_level") == "ERROR", 1).otherwise(0)).alias("error_logs"),
        sum(when(col("log_level") == "INFO", 1).otherwise(0)).alias("info_logs")
    )
    .withColumn("error_rate", round((col("error_logs") / col("total_logs")) * 100, 2))
)

In [0]:
display(fact_log_summary)

log_date,total_logs,error_logs,info_logs,error_rate
2026-04-05,80,16,64,20.0
2026-04-06,81,8,73,9.88
2026-03-30,58,6,52,10.34
2026-03-28,53,5,48,9.43
2026-03-31,49,5,44,10.2
2026-04-01,51,3,48,5.88
2026-04-02,56,3,53,5.36
2026-04-04,45,7,38,15.56
2026-03-29,52,7,45,13.46
2026-04-03,45,6,39,13.33


### creating hourly summary

In [0]:
fact_hourly_log_summary = (
    fact_log_detail.groupBy("log_date", "log_hour")
    .agg(
        count("*").alias("total_logs"),
        sum(when(col("log_level") == "ERROR", 1).otherwise(0)).alias("error_logs"),
        sum(when(col("log_level") == "INFO", 1).otherwise(0)).alias("info_logs")
    )
)

In [0]:
display(fact_hourly_log_summary)

log_date,log_hour,total_logs,error_logs,info_logs
2026-04-06,10,5,1,4
2026-04-05,16,8,1,7
2026-04-06,1,8,0,8
2026-04-06,14,5,0,5
2026-04-06,7,7,2,5
2026-04-05,15,5,0,5
2026-04-06,6,6,0,6
2026-04-05,19,6,3,3
2026-04-06,8,7,1,6
2026-04-05,20,5,1,4


### creating error message summary, error happen how many time in that day using group by

In [0]:
fact_error_message_summary = (
    fact_log_detail
    .filter(col("log_level") == "ERROR")
    .groupBy("log_date", "body_clean")
    .agg(count("*").alias("error_count"))
)

### creating user log summary

In [0]:
fact_user_log_summary = (
    fact_log_detail
    .filter(col("user_id").isNotNull())
    .groupBy("user_id")
    .agg(
        count("*").alias("total_logs"),
        sum(when(col("log_level") == "ERROR", 1).otherwise(0)).alias("error_logs"),
        sum(when(col("log_level") == "INFO", 1).otherwise(0)).alias("info_logs")
    )
    .withColumn("error_rate_%", round((col("error_logs") / col("total_logs")) * 100, 2))
    .withColumn("UpadtedDate", to_date(from_utc_timestamp(current_timestamp(), 'Asia/Kolkata')))
)

### Saving the tables in gold schemas

In [0]:
fact_log_detail.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("de_project.gold.fact_log_detail")

fact_log_summary.write.format("delta").mode("overwrite").saveAsTable("de_project.gold.fact_log_summary")

fact_hourly_log_summary.write.format("delta").mode("overwrite").saveAsTable("de_project.gold.fact_hourly_log_summary")

fact_error_message_summary.write.format("delta").mode("overwrite").saveAsTable("de_project.gold.fact_error_message_summary")

fact_user_log_summary.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("de_project.gold.fact_user_log_summary")